# QMIX Monolithic Training
This notebook runs the Monolithic QMIX baseline training on Google Colab using CPU only.

**Requirements:**
- Google Colab account
- Project files uploaded to Google Drive or cloned from repository
- SUMO traffic simulator (will be installed automatically)

**Training Setup:**
- Algorithm: QMIX (Monotonic Value Function Factorization)
- Environment: SUMO Traffic Simulation
- Device: CPU (compatible with local machine setup)
- Max Agents: 50 vehicles
- Episodes: 100 (configurable)

## 1. Environment Setup

First, we'll install SUMO (traffic simulator) and required dependencies.

In [1]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
else:
    print("Running locally")

Running on Google Colab


In [2]:
# Install SUMO on Colab (Ubuntu-based)
if IN_COLAB:
    print("Installing SUMO...")
    !apt-get update -qq
    !apt-get install -y -qq sumo sumo-tools sumo-doc
    
    # Set SUMO_HOME environment variable
    import os
    os.environ['SUMO_HOME'] = '/usr/share/sumo'
    
    print("SUMO installed successfully!")
    !sumo --version
else:
    print("Skipping SUMO installation (assuming already installed locally)")

Installing SUMO...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
SUMO installed successfully!
Eclipse SUMO sumo Version 1.12.0
 Build features: Linux-4.15.0-167-generic x86_64 GNU 11.2.0 None Proj GUI SWIG GDAL FFmpeg OSG GL2PS Eigen
 Copyright (C) 2001-2022 German Aerospace Center (DLR) and others; https://sumo.dlr.de

Eclipse SUMO sumo Version 1.12.0 is part of SUMO.
This program and the accompanying materials
are made available under the terms of the Eclipse Public License v2.0
which accompanies this distribution, and is available at
http://www.eclipse.org/legal/epl-v20.html
This program may also be made available under the following Secondary
Licenses when the conditions for such availability set forth in the Eclipse
Public License 2.0 are satisfied: GNU General Public License, version 2
or later which is available at
https://www.gnu.org/lic

In [ ]:
# Install Python dependencies
if IN_COLAB:
    print("Installing Python packages...")
    !pip install -q torch numpy matplotlib sumolib traci pyyaml tensorboard
    print("Dependencies installed!")
else:
    print("Skipping package installation (assuming dependencies already installed)")

Installing Python packages...
Dependencies installed!


## 2. Get Project Files

**Recommended: Clone from GitHub**
- Automatically downloads the latest version from GitHub
- No need to upload files manually

**Alternative: Use Google Drive**
- Useful if you have local modifications to test
- Requires manual upload to Drive

In [ ]:
# Option A: Clone from GitHub (Recommended)
if IN_COLAB:
    print("Cloning project from GitHub...")
    !git clone https://github.com/Akiri017/apriori_civiq.git
    
    import os
    PROJECT_PATH = '/content/apriori_civiq'
    os.chdir(PROJECT_PATH)
    print(f"Project cloned successfully to: {PROJECT_PATH}")
    print(f"Working directory: {os.getcwd()}")
else:
    # Running locally - assume we're already in the right directory
    import os
    PROJECT_PATH = os.getcwd()
    print(f"Working directory: {PROJECT_PATH}")

Cloning project from GitHub...
fatal: destination path 'apriori_civiq' already exists and is not an empty directory.
✓ Project cloned successfully to: /content/apriori_civiq
✓ Working directory: /content/apriori_civiq


In [ ]:
# Option B: Mount Google Drive (alternative - uncomment if needed)
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     
#     # Update this path to your Drive folder
#     PROJECT_PATH = '/content/drive/MyDrive/apriori_civiq'
#     
#     import os
#     if os.path.exists(PROJECT_PATH):
#         os.chdir(PROJECT_PATH)
#         print(f"Working directory: {os.getcwd()}")
#     else:
#         print(f"ERROR: Path not found: {PROJECT_PATH}")
#         print("Update PROJECT_PATH to match your Google Drive folder structure")

## 3. Verify Project Structure

Let's verify that all necessary files are present.

In [ ]:
import os

# Check key files and directories
required_paths = [
    'src/main.py',
    'src/baseline/mono_qmix/',
    'scenarios/test_simple/',
]

print("Checking project structure...")
print(f"Current directory: {os.getcwd()}\n")

missing = []
for path in required_paths:
    full_path = os.path.join(PROJECT_PATH, path)
    exists = os.path.exists(full_path)
    status = "FOUND" if exists else "MISSING"
    print(f"{status} {path}")
    if not exists:
        missing.append(path)

if missing:
    print(f"\nWarning: Missing {len(missing)} required paths")
    print("\nTroubleshooting:")
    print("1. Verify PROJECT_PATH is correct in the previous cell")
    print("2. Make sure you uploaded the entire project folder to Google Drive")
    print("3. Check that the folder contains the 'src' and 'scenarios' directories")
else:
    print("\nAll required files found.")

Checking project structure...
Current directory: /content/apriori_civiq

✓ src/main.py
✓ src/baseline/mono_qmix/
✓ scenarios/test_simple/

✓ All required files found!


## 4. Configure Training Parameters

Modify these parameters based on your needs. For Colab, we use CPU-only and disable GUI.

In [ ]:
# Training Configuration
TRAINING_CONFIG = {
    # Environment
    'sumo_config': 'scenarios/test_simple/Configuration_med.sumocfg',
    'max_agents': 20,
    'use_gui': False,  # Must be False for Colab (no display)
    
    # Training hyperparameters
    'episodes': 50,  # Reduced for initial testing (was 100)
    'batch_size': 16,  # Reduced for stability (was 32)
    'buffer_capacity': 2000,  # Reduced for memory (was 5000)
    'max_episode_length': 500,  # Reduced for stability (was 3600)
    
    # Learning parameters
    'learning_rate': 0.0005,
    'gamma': 0.99,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay': 0.995,
    
    # Updates
    'target_update_freq': 200,
    'min_buffer_size': 64,  # Reduced (was 128)
    
    # Device
    'device': 'cpu',  # CPU-only for consistency with local training
    
    # Monitoring
    'verbose': True,  # Print detailed step information
    'step_interval': 50,  # Print status every N steps during episode
}

print("Training Configuration (Optimized for Colab):")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")
print("\n⚠️ Note: Config optimized for stability on Colab")
print("   Increase episodes/agents/length after successful test run")

Training Configuration (Optimized for Colab):
  sumo_config: scenarios/test_simple/Configuration_med.sumocfg
  max_agents: 20
  use_gui: False
  episodes: 50
  batch_size: 16
  buffer_capacity: 2000
  max_episode_length: 500
  learning_rate: 0.0005
  gamma: 0.99
  epsilon_start: 1.0
  epsilon_end: 0.05
  epsilon_decay: 0.995
  target_update_freq: 200
  min_buffer_size: 64
  device: cpu
  verbose: True
  step_interval: 50

⚠️ Note: Config optimized for stability on Colab
   Increase episodes/agents/length after successful test run


## 5. Import Required Modules

Import the QMIX implementation and supporting libraries.

In [8]:
# Add src to path
import sys
import os
sys.path.insert(0, os.path.join(PROJECT_PATH, 'src'))

# Import training modules
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import datetime

from baseline.mono_qmix import QMIX_Controller, QMIXSumoEnv
from baseline.mono_qmix.replay_buffer import EpisodeReplayBuffer

print("✓ Modules imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {TRAINING_CONFIG['device']}")

✓ Modules imported successfully!
PyTorch version: 2.10.0+cpu
Device: cpu


## 6. Define Training Function

This is adapted from the main training script, optimized for notebook execution.

In [9]:
# Quick SUMO test to catch configuration errors early
print("Testing SUMO environment setup...")

# First, close any existing SUMO connections
try:
    import traci
    if traci.isLoaded():
        print("Closing existing SUMO connection...")
        traci.close()
        print("✓ Previous connection closed")
except:
    pass

try:
    # Create environment (this may start SUMO in __init__)
    test_env = QMIXSumoEnv(
        sumo_config=TRAINING_CONFIG['sumo_config'],
        max_agents=TRAINING_CONFIG['max_agents'],
        use_gui=False
    )
    
    print("✓ SUMO initialized successfully")
    
    # Close and reset to test the full cycle
    print("Testing environment reset cycle...")
    test_env.close()
    
    state, obs = test_env.reset()
    print(f"✓ State shape: {state.shape}")
    print(f"✓ Observation shape: {obs.shape}")
    
    print("Testing 10 simulation steps...")
    for i in range(10):
        # Random actions for all agents
        actions = np.random.randint(0, test_env.n_actions, size=test_env.max_agents)
        try:
            next_state, next_obs, reward, done = test_env.step(actions)
            if i % 5 == 0:
                print(f"  Step {i+1}: reward={reward:.2f}, done={done}")
        except Exception as e:
            print(f"❌ Error at step {i+1}: {e}")
            break
    
    print("✓ Test steps completed")
    
    # Clean up
    test_env.close()
    print("✓ Test environment closed")
    print("\n✅ SUMO test passed! Ready for training.")
    
except Exception as e:
    print(f"\n❌ SUMO test failed: {e}")
    print("\nTroubleshooting:")
    print("1. Run the 'Close Stray SUMO Connections' cell above")
    print("2. Check that the SUMO config file exists and is valid")
    print("3. Verify SUMO_HOME is set correctly")
    print("4. Restart the kernel if connection issues persist")
    import traceback
    traceback.print_exc()
    
    # Attempt cleanup
    try:
        import traci
        if traci.isLoaded():
            traci.close()
            print("\n✓ Cleaned up SUMO connection")
    except:
        pass

Testing SUMO environment setup...
Network topology loaded: 16 edges with routing options
✓ SUMO initialized successfully
Testing environment reset cycle...
 Retrying in 1 seconds
✓ State shape: (341,)
✓ Observation shape: (20, 16)
Testing 10 simulation steps...
  Step 1: reward=-0.00, done=False
  Step 6: reward=-0.00, done=False
✓ Test steps completed
✓ Test environment closed

✅ SUMO test passed! Ready for training.


## 5.5. Test SUMO Connection (Recommended)

**Important:** If you get "Connection already active" errors, run the helper cell below FIRST to close stray connections, then re-run this test cell.

## 5.6. Diagnose Vehicle Spawning Issue (If Seeing "0 Active Vehicles")

**Run this diagnostic cell if you see "Active vehicles: 0" in all steps.**

This will help identify why vehicles aren't appearing in the simulation.

In [10]:
import os

# Close any existing connections first
try:
    import traci
    if traci.isLoaded():
        traci.close()
except:
    pass

print("=== VEHICLE SPAWNING DIAGNOSTIC ===\n")

# 1. Check if SUMO config file exists
sumo_config = TRAINING_CONFIG['sumo_config']
print(f"1. Checking SUMO config: {sumo_config}")
if os.path.exists(sumo_config):
    print(f"   ✓ Config file exists")
else:
    print(f"   ✗ Config file NOT FOUND!")
    print(f"   Current directory: {os.getcwd()}")

# 2. Read the config to see what route file it references  
print("\n2. Reading config file...")
try:
    with open(sumo_config, 'r') as f:
        config_content = f.read()
        # Find route-files line
        import re
        route_match = re.search(r'<route-files\s+value="([^"]+)"', config_content)
        if route_match:
            route_file = route_match.group(1)
            print(f"   Route file specified: {route_file}")
            
            # Check if it's a trip file or route file
            if 'trip' in route_file.lower():
                print(f"   ⚠️  WARNING: Using TRIP file (needs routing)")
                print(f"   Trip files contain origin-destination pairs, not full routes")
            else:
                print(f"   ✓ Using route file (pre-routed)")
                
            # Check if file exists
            config_dir = os.path.dirname(sumo_config)
            full_route_path = os.path.join(config_dir, route_file)
            if os.path.exists(full_route_path):
                print(f"   ✓ Route file exists: {full_route_path}")
                
                # Count vehicles in file
                with open(full_route_path, 'r') as rf:
                    content = rf.read()
                    trip_count = content.count('<trip ')
                    vehicle_count = content.count('<vehicle ')
                    print(f"   Found: {trip_count} trips, {vehicle_count} routed vehicles")
                    
                    if trip_count > 0 and vehicle_count == 0:
                        print(f"   ⚠️  PROBLEM IDENTIFIED: File has trips but no routes!")
                        print(f"   Solution: Need to route trips or use pre-routed file")
            else:
                print(f"   ✗ Route file NOT FOUND: {full_route_path}")
except Exception as e:
    print(f"   Error reading config: {e}")

# 3. Test SUMO startup and check vehicle loading
print("\n3. Starting SUMO to test vehicle loading...")
try:
    import sumolib
    sumo_binary = sumolib.checkBinary('sumo')  # Use non-GUI
    
    traci.start([
        sumo_binary,
        "-c", sumo_config,
        "--no-step-log", "true",
        "--time-to-teleport", "-1"
    ])
    
    print("   ✓ SUMO started successfully")
    
    # Check loaded vehicles
    loaded_vehicles = traci.simulation.getLoadedNumber()
    print(f"\n4. Vehicles loaded in simulation: {loaded_vehicles}")
    
    if loaded_vehicles == 0:
        print("   ✗ CRITICAL: No vehicles loaded!")
        print("   Possible causes:")
        print("   - Trip file needs routing (use duarouter)")
        print("   - Route file is empty")
        print("   - Vehicles have invalid routes")
    else:
        print(f"   ✓ {loaded_vehicles} vehicles are loaded")
    
    # Step through first 50 simulation steps and count vehicles
    print("\n5. Stepping through first 50 steps to check vehicle spawning...")
    for i in range(50):
        traci.simulationStep()
        veh_count = len(traci.vehicle.getIDList())
        if veh_count > 0:
            print(f"   Step {i+1}: {veh_count} vehicles active")
            if i < 5:  # Show first few vehicle IDs
                print(f"      Vehicle IDs: {traci.vehicle.getIDList()[:5]}")
            break
    else:
        print("   ✗ PROBLEM: No vehicles appeared in first 50 steps")
        print("   Check that vehicle departure times are early enough")
    
    traci.close()
    print("\n✓ Diagnostic complete")
    
except Exception as e:
    print(f"   ✗ Error during diagnostic: {e}")
    import traceback
    traceback.print_exc()
    try:
        traci.close()
    except:
        pass

print("\n=== DIAGNOSTIC COMPLETE ===")
print("\nIf the problem is 'trips need routing', see the fix in the next cell.")

=== VEHICLE SPAWNING DIAGNOSTIC ===

1. Checking SUMO config: scenarios/test_simple/Configuration_med.sumocfg
   ✓ Config file exists

2. Reading config file...
   Route file specified: trips_med.xml
   ⚠️  WARNING: Using TRIP file (needs routing)
   Trip files contain origin-destination pairs, not full routes
   ✓ Route file exists: scenarios/test_simple/trips_med.xml
   Found: 1126 trips, 0 routed vehicles
   ⚠️  PROBLEM IDENTIFIED: File has trips but no routes!
   Solution: Need to route trips or use pre-routed file

3. Starting SUMO to test vehicle loading...
 Retrying in 1 seconds
   ✓ SUMO started successfully

4. Vehicles loaded in simulation: 2
   ✓ 2 vehicles are loaded

5. Stepping through first 50 steps to check vehicle spawning...
   Step 1: 1 vehicles active
      Vehicle IDs: ('0',)

✓ Diagnostic complete

=== DIAGNOSTIC COMPLETE ===

If the problem is 'trips need routing', see the fix in the next cell.


## 5.7. FIX: Route the Trip File or Use Pre-Routed File

**Choose ONE of the solutions below based on the diagnostic results.**

In [11]:
# SOLUTION 1: Convert trips to routes using duarouter (Recommended)
print("=== CONVERTING TRIPS TO ROUTES ===\n")

import os
import subprocess

# Get the scenario directory
config_path = TRAINING_CONFIG['sumo_config']
scenario_dir = os.path.dirname(config_path)

# Input/output files
trip_file = os.path.join(scenario_dir, "trips_med.xml")
net_file = os.path.join(scenario_dir, "final_map.net.xml")
output_route_file = os.path.join(scenario_dir, "routes_med.rou.xml")

print(f"Input trip file: {trip_file}")
print(f"Network file: {net_file}")
print(f"Output route file: {output_route_file}")

# Check if files exist
if not os.path.exists(trip_file):
    print(f"\n✗ ERROR: Trip file not found: {trip_file}")
elif not os.path.exists(net_file):
    print(f"\n✗ ERROR: Network file not found: {net_file}")
else:
    print("\n✓ Files found, running duarouter...")
    
    try:
        # Run duarouter to convert trips to routes
        cmd = [
            "duarouter",
            "-n", net_file,
            "-r", trip_file,
            "-o", output_route_file,
            "--ignore-errors",
            "--no-warnings",
            "--no-step-log"
        ]
        
        print(f"Command: {' '.join(cmd)}\n")
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print("✓ Routes generated successfully!")
            print(f"✓ Output file: {output_route_file}")
            
            # Update training config to use the new route file
            print("\n✓ Updating training configuration...")
            
            # Create a modified config file
            new_config_path = os.path.join(scenario_dir, "Configuration_med_routed.sumocfg")
            
            with open(config_path, 'r') as f:
                config_content = f.read()
            
            # Replace trips_med.xml with routes_med.rou.xml
            config_content = config_content.replace('trips_med.xml', 'routes_med.rou.xml')
            
            with open(new_config_path, 'w') as f:
                f.write(config_content)
            
            print(f"✓ Created new config: {new_config_path}")
            
            # Update the training configuration
            TRAINING_CONFIG['sumo_config'] = new_config_path
            print(f"✓ Training config updated to use routed file")
            
            print("\n✅ READY TO TRAIN! Re-run the SUMO test cell (section 5.5) to verify.")
            
        else:
            print(f"✗ duarouter failed with return code {result.returncode}")
            print(f"Error output:\n{result.stderr}")
            
    except FileNotFoundError:
        print("✗ ERROR: duarouter not found. Make sure SUMO is installed correctly.")
        print("On Colab, duarouter should be installed with SUMO.")
    except Exception as e:
        print(f"✗ ERROR: {e}")
        import traceback
        traceback.print_exc()

=== CONVERTING TRIPS TO ROUTES ===

Input trip file: scenarios/test_simple/trips_med.xml
Network file: scenarios/test_simple/final_map.net.xml
Output route file: scenarios/test_simple/routes_med.rou.xml

✓ Files found, running duarouter...
Command: duarouter -n scenarios/test_simple/final_map.net.xml -r scenarios/test_simple/trips_med.xml -o scenarios/test_simple/routes_med.rou.xml --ignore-errors --no-warnings --no-step-log

✓ Routes generated successfully!
✓ Output file: scenarios/test_simple/routes_med.rou.xml

✓ Updating training configuration...
✓ Created new config: scenarios/test_simple/Configuration_med_routed.sumocfg
✓ Training config updated to use routed file

✅ READY TO TRAIN! Re-run the SUMO test cell (section 5.5) to verify.


In [12]:
# SOLUTION 2: Use existing pre-routed file (Quick Fix)
# Use this if routes.rou.xml already exists in the scenario folder

print("=== USING EXISTING ROUTED FILE ===\n")

import os

config_path = TRAINING_CONFIG['sumo_config']
scenario_dir = os.path.dirname(config_path)

# Check for existing route files
existing_route_file = os.path.join(scenario_dir, "routes.rou.xml")

if os.path.exists(existing_route_file):
    print(f"✓ Found existing route file: {existing_route_file}")
    
    # Count vehicles in the file
    with open(existing_route_file, 'r') as f:
        content = f.read()
        vehicle_count = content.count('<vehicle ')
        print(f"✓ Contains {vehicle_count} routed vehicles")
    
    if vehicle_count > 0:
        # Create modified config file
        new_config_path = os.path.join(scenario_dir, "Configuration_med_fixed.sumocfg")
        
        with open(config_path, 'r') as f:
            config_content = f.read()
        
        # Replace trip file with route file
        config_content = config_content.replace('trips_med.xml', 'routes.rou.xml')
        config_content = config_content.replace('trips_low.xml', 'routes.rou.xml')
        config_content = config_content.replace('trips_high.xml', 'routes.rou.xml')
        
        with open(new_config_path, 'w') as f:
            f.write(config_content)
        
        print(f"✓ Created new config: {new_config_path}")
        
        # Update training config
        TRAINING_CONFIG['sumo_config'] = new_config_path
        print(f"✓ Training config updated")
        
        print("\n✅ READY TO TRAIN! Re-run the SUMO test cell (section 5.5) to verify.")
        print(f"\n⚠️  Note: routes.rou.xml was generated from trips_high.xml")
        print("   This means it has HIGH traffic density (~2.5s between vehicles)")
        print("   Consider running duarouter on trips_med.xml for medium traffic")
    else:
        print("✗ Route file exists but contains no vehicles")
else:
    print(f"✗ No pre-routed file found at: {existing_route_file}")
    print("   Use Solution 1 (duarouter) instead")

=== USING EXISTING ROUTED FILE ===

✓ Found existing route file: scenarios/test_simple/routes.rou.xml
✓ Contains 1407 routed vehicles
✓ Created new config: scenarios/test_simple/Configuration_med_fixed.sumocfg
✓ Training config updated

✅ READY TO TRAIN! Re-run the SUMO test cell (section 5.5) to verify.

⚠️  Note: routes.rou.xml was generated from trips_high.xml
   This means it has HIGH traffic density (~2.5s between vehicles)
   Consider running duarouter on trips_med.xml for medium traffic


In [13]:
# Force close any active SUMO/TraCI connections
import traci
import sys

try:
    if traci.isLoaded():
        print("Closing active SUMO connection...")
        traci.close()
        print("✓ SUMO connection closed")
    else:
        print("No active SUMO connection found")
except Exception as e:
    print(f"Attempted to close connection: {e}")
    
# Also try to close through connection class
try:
    if hasattr(traci, '_connections'):
        for label in list(traci._connections.keys()):
            try:
                traci.switch(label)
                traci.close()
                print(f"✓ Closed connection: {label}")
            except:
                pass
except:
    pass

print("✓ Cleanup complete - ready for fresh start")

No active SUMO connection found
✓ Cleanup complete - ready for fresh start


### Helper: Close Stray SUMO Connections

If you see "Connection already active" errors, run this cell to force close all SUMO connections.

In [14]:
def train_qmix(config):
    """Train Monolithic QMIX with episodic replay buffer and BPTT."""
    
    # Initialize Environment
    print("Initializing SUMO environment...")
    env = QMIXSumoEnv(
        sumo_config=config['sumo_config'],
        max_agents=config['max_agents'],
        use_gui=config['use_gui']
    )
    
    # Initialize Controller
    print("Initializing QMIX controller...")
    controller = QMIX_Controller(
        n_agents=config['max_agents'],
        state_shape=env.state_shape,
        obs_shape=env.obs_shape,
        n_actions=env.n_actions,
        lr=config['learning_rate'],
        gamma=config['gamma'],
        device=config['device']
    )
    
    # Initialize Episode Replay Buffer
    print("Initializing replay buffer...")
    replay_buffer = EpisodeReplayBuffer(
        capacity=config['buffer_capacity'],
        max_seq_len=config['max_episode_length'],
        state_shape=env.state_shape,
        obs_shape=env.obs_shape,
        n_agents=config['max_agents']
    )
    
    # Training state
    epsilon = config['epsilon_start']
    rewards_history = []
    loss_history = []
    verbose = config.get('verbose', False)
    step_interval = config.get('step_interval', 100)
    
    print("\n" + "="*60)
    print("Starting Monolithic QMIX Training")
    print("="*60)
    print(f"Observation shape: {env.obs_shape}D")
    print(f"State shape: {env.state_shape}D")
    print(f"Action space: {env.n_actions}")
    print(f"Max agents: {config['max_agents']}")
    print(f"Episodes: {config['episodes']}")
    print(f"Verbose mode: {verbose}")
    print("="*60 + "\n")
    
    try:
        for ep in range(config['episodes']):
            # ===== EPISODE DATA COLLECTION =====
            if verbose:
                print(f"\n[Episode {ep+1}] Resetting environment...")
            
            try:
                state, obs = env.reset()
            except Exception as e:
                print(f"❌ Error resetting environment: {e}")
                print("Attempting to restart SUMO...")
                env.close()
                env = QMIXSumoEnv(
                    sumo_config=config['sumo_config'],
                    max_agents=config['max_agents'],
                    use_gui=config['use_gui']
                )
                state, obs = env.reset()
                print("✓ Environment restarted successfully")
            
            hidden = controller.init_hidden(batch_size=1)
            
            # Storage for current episode
            episode_data = {
                'obs': [],
                'state': [],
                'actions': [],
                'rewards': [],
                'dones': []
            }
            
            # Store initial observation and state
            episode_data['obs'].append(obs)
            episode_data['state'].append(state)
            
            episode_reward = 0
            done = False
            t = 0
            
            if verbose:
                print(f"[Episode {ep+1}] Starting simulation...")
            
            while not done and t < config['max_episode_length']:
                try:
                    # Select actions with current hidden state
                    actions, hidden = controller.select_actions(obs, hidden, epsilon)
                    
                    # Environment step
                    next_state, next_obs, reward, done = env.step(actions)
                    
                    # Store transition
                    episode_data['actions'].append(actions)
                    episode_data['rewards'].append([reward])
                    episode_data['dones'].append([float(done)])
                    
                    # Store next observation and state
                    episode_data['obs'].append(next_obs)
                    episode_data['state'].append(next_state)
                    
                    # Update for next iteration
                    obs = next_obs
                    state = next_state
                    episode_reward += reward
                    t += 1
                    
                    # Periodic status update
                    if verbose and t % step_interval == 0:
                        print(f"  Step {t:4d} | Reward: {episode_reward:7.2f} | Active vehicles: {np.count_nonzero(obs.sum(axis=1))}")
                
                except Exception as e:
                    print(f"\n❌ Error at step {t}: {e}")
                    print(f"   Episode {ep+1} terminated early")
                    done = True
                    break
            
            if verbose:
                print(f"[Episode {ep+1}] Completed with {t} steps, reward: {episode_reward:.2f}")
            
            # ===== STORE EPISODE IN REPLAY BUFFER =====
            if t > 0:  # Only store if episode had at least one step
                episode = {
                    'obs': np.array(episode_data['obs']),
                    'state': np.array(episode_data['state']),
                    'actions': np.array(episode_data['actions']),
                    'rewards': np.array(episode_data['rewards']),
                    'dones': np.array(episode_data['dones'])
                }
                
                replay_buffer.add_episode(episode)
                rewards_history.append(episode_reward)
            else:
                print(f"⚠️ Episode {ep+1} had no steps, skipping...")
                continue
            
            # ===== TRAINING STEP =====
            mean_loss = 0.0
            if len(replay_buffer) >= config['min_buffer_size']:
                batch = replay_buffer.sample(config['batch_size'])
                loss = controller.train(batch)
                mean_loss = loss
                loss_history.append(loss)
            
            # Decay Epsilon
            epsilon = max(config['epsilon_end'], epsilon * config['epsilon_decay'])
            
            # ===== UPDATE TARGET NETWORKS =====
            if (ep + 1) % config['target_update_freq'] == 0:
                controller.update_targets()
                print(f"  → Target networks updated at episode {ep+1}")
            
            # Logging
            if (ep + 1) % 10 == 0 or ep == 0:
                avg_reward = np.mean(rewards_history[-10:]) if len(rewards_history) >= 10 else episode_reward
                print(f"Episode {ep+1:4d}/{config['episodes']} | "
                      f"Steps: {t:4d} | "
                      f"Reward: {episode_reward:7.2f} | "
                      f"Avg(10): {avg_reward:7.2f} | "
                      f"Loss: {mean_loss:6.4f} | "
                      f"ε: {epsilon:.3f} | "
                      f"Buffer: {len(replay_buffer):4d}")
    
    except KeyboardInterrupt:
        print("\n⚠️ Training interrupted by user.")
    
    except Exception as e:
        print(f"\n❌ Critical error during training: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        # Always close environment cleanly
        print("\nClosing environment...")
        try:
            env.close()
            print("✓ Environment closed successfully.")
        except:
            print("⚠️ Environment already closed or error closing")
    
    # Return training results
    return {
        'controller': controller,
        'rewards_history': rewards_history,
        'loss_history': loss_history,
        'config': config
    }

## 7. Run Training

Execute the training process. This may take a while depending on the number of episodes.

In [15]:
# Run training
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

results = train_qmix(TRAINING_CONFIG)

print(f"\nEnd time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Training completed!")

Start time: 2026-03-04 06:10:25

Initializing SUMO environment...
Network topology loaded: 16 edges with routing options
Initializing QMIX controller...
Initializing replay buffer...

Starting Monolithic QMIX Training
Observation shape: 16D
State shape: 341D
Action space: 6
Max agents: 20
Episodes: 50
Verbose mode: True


[Episode 1] Resetting environment...
 Retrying in 1 seconds
[Episode 1] Starting simulation...
  Step   50 | Reward:   -0.35 | Active vehicles: 0
  Step  100 | Reward:   -0.75 | Active vehicles: 0
  Step  150 | Reward:   -1.06 | Active vehicles: 0
  Step  200 | Reward:   -2.20 | Active vehicles: 0
  Step  250 | Reward:   -3.24 | Active vehicles: 0
  Step  300 | Reward:   -3.87 | Active vehicles: 0
  Step  350 | Reward:   -4.35 | Active vehicles: 0
  Step  400 | Reward:   -5.20 | Active vehicles: 0
  Step  450 | Reward:   -6.62 | Active vehicles: 0
  Step  500 | Reward:   -7.32 | Active vehicles: 0
[Episode 1] Completed with 500 steps, reward: -7.32
Episode    1/50 | S

Traceback (most recent call last):
  File "/tmp/ipykernel_18776/2395969831.py", line 59, in train_qmix
    state, obs = env.reset()
                 ^^^^^^^^^^^
  File "/content/apriori_civiq/src/baseline/mono_qmix/env.py", line 67, in reset
    traci.start([
  File "/usr/local/lib/python3.12/dist-packages/traci/main.py", line 139, in start
    raise TraCIException("Connection '%s' is already active." % label)
traci.exceptions.TraCIException: Connection 'default' is already active.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_18776/2395969831.py", line 69, in train_qmix
    state, obs = env.reset()
                 ^^^^^^^^^^^
  File "/content/apriori_civiq/src/baseline/mono_qmix/env.py", line 67, in reset
    traci.start([
  File "/usr/local/lib/python3.12/dist-packages/traci/main.py", line 139, in start
    raise TraCIException("Connection '%s' is already active." % label)
traci.exceptions.TraCIExcepti

## 5.7. FIX: Generate Initial Routes for Dynamic QMIX Routing

**Understanding the Issue:**
- Trip files (`trips_med.xml`) contain only origin-destination pairs, not routes
- Vehicles need an initial route to spawn in SUMO  
- QMIX will then dynamically reroute them based on traffic conditions

**The Solution:**
Generate simple initial routes (shortest path), then enable QMIX to change them during training.

In [ ]:
# Generate initial routes from trip file
import subprocess
import os

print("="*60)
print("  GENERATING INITIAL ROUTES FROM TRIPS")
print("="*60)

scenario_dir = "scenarios/test_simple"
traffic_level = "med"  # Change to "low" or "high" if needed

# File paths
net_file = f"{scenario_dir}/final_map.net.xml"
trip_file = f"{scenario_dir}/trips_{traffic_level}.xml"
output_route_file = f"{scenario_dir}/routes_{traffic_level}.rou.xml"

print(f"\nInput:  {trip_file}")
print(f"Output: {output_route_file}\n")

try:
    # Run duarouter to generate initial routes
    result = subprocess.run([
        "duarouter",
        "-n", net_file,
        "-r", trip_file,
        "-o", output_route_file,
        "--ignore-errors",
        "--no-warnings",
        "--no-step-log"
    ], capture_output=True, text=True, check=True)
    
    print("✅ Routes generated successfully!")
    
    # Count vehicles
    with open(output_route_file, 'r') as f:
        route_count = f.read().count('<vehicle ')
    print(f"✅ Created {route_count} vehicle routes")
    
    # Create updated config file
    config_file = f"{scenario_dir}/Configuration_{traffic_level}.sumocfg"
    new_config = f"{scenario_dir}/Configuration_{traffic_level}_routed.sumocfg"
    
    with open(config_file, 'r') as f:
        config_content = f.read()
    
    # Replace trip file with route file
    config_content = config_content.replace(
        f'trips_{traffic_level}.xml',
        f'routes_{traffic_level}.rou.xml'
    )
    
    with open(new_config, 'w') as f:
        f.write(config_content)
    
    print(f"✅ Created updated config: Configuration_{traffic_level}_routed.sumocfg")
    
    # Update training configuration
    TRAINING_CONFIG['sumo_config'] = new_config
    print(f"\n✅ Training config updated to use routed file")
    print(f"\n📝 Note: These are simple shortest-path routes")
    print(f"   QMIX can dynamically reroute vehicles during training")
    print(f"\n✅ READY TO TRAIN! Re-run the SUMO test (section 5.5) to verify.")
    
except subprocess.CalledProcessError as e:
    print(f"❌ ERROR: duarouter failed")
    print(f"   {e.stderr}")
except FileNotFoundError:
    print(f"❌ ERROR: duarouter not found")
    print(f"   Make sure SUMO is installed (should be from earlier cells)")
except Exception as e:
    print(f"❌ ERROR: {e}")

In [ ]:
# Save models
if len(results['rewards_history']) > 0:
    # Create output directory
    save_dir = os.path.join(PROJECT_PATH, 'outputs', 'models')
    os.makedirs(save_dir, exist_ok=True)
    
    # Save agent and mixer networks
    controller = results['controller']
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    agent_path = os.path.join(save_dir, f"qmix_agent_colab_{timestamp}.pth")
    mixer_path = os.path.join(save_dir, f"qmix_mixer_colab_{timestamp}.pth")
    
    torch.save(controller.agent_net.state_dict(), agent_path)
    torch.save(controller.mixer.state_dict(), mixer_path)
    
    print("✓ Models saved:")
    print(f"  Agent:  {agent_path}")
    print(f"  Mixer:  {mixer_path}")
    
    # Save training history
    history_path = os.path.join(save_dir, f"training_history_{timestamp}.npz")
    np.savez(history_path,
             rewards=results['rewards_history'],
             losses=results['loss_history'])
    print(f"  History: {history_path}")
    
    # Save configuration
    import json
    config_path = os.path.join(save_dir, f"config_{timestamp}.json")
    with open(config_path, 'w') as f:
        json.dump(TRAINING_CONFIG, f, indent=2)
    print(f"  Config:  {config_path}")
    
else:
    print("⚠️ No models to save (training did not complete)")

⚠️ No models to save (training did not complete)


## 10. Download Results (Optional)

If you want to download the models to your local machine, use the code below.

In [ ]:
# Download files to local machine (Colab only)
if IN_COLAB:
    from google.colab import files
    
    print("Download trained models?")
    print("Uncomment the lines below to download:")
    
    # Uncomment to download
    # files.download(agent_path)
    # files.download(mixer_path)
    # files.download(history_path)
    # files.download(config_path)
else:
    print("Running locally - files already saved to disk")

Download trained models?
Uncomment the lines below to download:


## 11. Test Trained Model (Optional)

Run a test episode with the trained model to see its performance.

In [ ]:
# Test the trained model
def test_model(controller, config, num_episodes=5):
    """Run test episodes with trained model."""
    print("\nTesting trained model...")
    
    # Initialize environment (no GUI)
    env = QMIXSumoEnv(
        sumo_config=config['sumo_config'],
        max_agents=config['max_agents'],
        use_gui=False
    )
    
    test_rewards = []
    
    try:
        for ep in range(num_episodes):
            state, obs = env.reset()
            hidden = controller.init_hidden(batch_size=1)
            
            episode_reward = 0
            done = False
            t = 0
            
            # Run episode with greedy policy (epsilon=0)
            while not done and t < config['max_episode_length']:
                actions, hidden = controller.select_actions(obs, hidden, epsilon=0.0)
                next_state, next_obs, reward, done = env.step(actions)
                
                obs = next_obs
                state = next_state
                episode_reward += reward
                t += 1
            
            test_rewards.append(episode_reward)
            print(f"  Test Episode {ep+1}/{num_episodes}: Reward = {episode_reward:.2f}, Steps = {t}")
    
    finally:
        env.close()
    
    print(f"\nTest Results:")
    print(f"  Mean Reward: {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")
    print(f"  Min Reward:  {np.min(test_rewards):.2f}")
    print(f"  Max Reward:  {np.max(test_rewards):.2f}")
    
    return test_rewards

# Uncomment to run tests
# if len(results['rewards_history']) > 0:
#     test_rewards = test_model(results['controller'], TRAINING_CONFIG)

## Summary

This notebook provides a complete pipeline for training the Monolithic QMIX baseline on Google Colab using CPU.

**Key Features:**
- ✓ CPU-only training (compatible with local setup)
- ✓ Automatic SUMO installation
- ✓ GitHub integration (auto-clone latest version)
- ✓ **Verbose logging** - See step-by-step progress
- ✓ **Error recovery** - Automatic SUMO restart on crashes
- ✓ **Progress monitoring** - Real-time status updates during episodes
- ✓ **Pre-flight test** - Validate SUMO before training
- ✓ Model checkpointing and visualization

**Recommended Workflow:**
1. Run all setup cells (1-5)
2. **Run the SUMO test cell (5.5)** to catch issues early
3. Adjust `TRAINING_CONFIG` if test fails (see Troubleshooting section)
4. Run training (section 7)
5. Monitor verbose output to track progress
6. Visualize results and save models

**Performance Tips:**
- Start with smaller episodes (500-1000 steps) and fewer agents (20-30)
- Enable verbose mode to see what's happening
- Use the troubleshooting section if you encounter errors
- Gradually increase complexity after successful initial runs

**Next Steps:**
1. Experiment with different hyperparameters
2. Try different traffic scenarios (change `sumo_config`)
3. Increase episodes/complexity after validation
4. Compare with other baselines or hierarchical approaches

## Troubleshooting SUMO Errors

**Common issues and solutions:**

### 1. "Retrying in 1 seconds" or TraCI communication errors
**Cause:** SUMO crashed or became unresponsive during simulation

**Solutions:**
- ✓ Enable `verbose=True` in training config to see progress
- ✓ Reduce `max_episode_length` (try 1000 instead of 3600)
- ✓ Use simpler scenario (`test_simple` instead of `bgc_full`)
- ✓ Reduce `max_agents` (try 20-30 instead of 50)
- ✓ Run the SUMO test cell first to catch config issues

### 2. "unpack requires a buffer" error
**Cause:** SUMO sent corrupted data, usually means it crashed

**Solutions:**
- Check SUMO scenario files are valid (`.sumocfg`, `.net.xml`, `.rou.xml`)
- Ensure routes don't have impossible paths
- Try running SUMO manually: `sumo -c scenarios/test_simple/Configuration_med.sumocfg --step-length 1`

### 3. Training is slow or hangs
**Cause:** Long episodes with many vehicles

**Solutions:**
- Reduce `max_episode_length` to 500-1000 steps
- Reduce `max_agents` to 20-30
- Set `step_interval=50` to see progress more frequently
- Use smaller traffic scenario

### 4. Out of memory errors
**Solutions:**
- Reduce `buffer_capacity` to 1000-2000
- Reduce `batch_size` to 16
- Reduce `max_episode_length` to 500
- Reduce `max_agents` to 20

### Quick Fix Configuration (if having issues):
```python
TRAINING_CONFIG = {
    'sumo_config': 'scenarios/test_simple/Configuration_med.sumocfg',
    'max_agents': 20,  # Reduced from 50
    'use_gui': False,
    'episodes': 50,  # Reduced from 100
    'batch_size': 16,  # Reduced from 32
    'buffer_capacity': 2000,  # Reduced from 5000
    'max_episode_length': 500,  # Reduced from 3600
    'learning_rate': 0.0005,
    'gamma': 0.99,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay': 0.995,
    'target_update_freq': 200,
    'min_buffer_size': 64,  # Reduced from 128
    'device': 'cpu',
    'verbose': True,  # Enable detailed logging
    'step_interval': 50,  # Status update every 50 steps
}
```